In [19]:
import torch
import torch.nn as nn
import numpy as np

In [10]:
print("PyTorch version:", torch.__version__)

PyTorch version: 2.5.1+cu121


In [11]:
# We are now starting Decoder


In [12]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, dimensions, seq_len):
        super(Decoder, self).__init__()
        self.vocab_size = vocab_size
        self.dimensions = dimensions
        self.seq_len = seq_len
        self.embedding = nn.Embedding(vocab_size, dimensions)
        self.pos_embedding = nn.Embedding(seq_len, dimensions)

        self.query = nn.Linear(dimensions, dimensions)
        self.key = nn.Linear(dimensions, dimensions)    
        self.value = nn.Linear(dimensions, dimensions)

        self.w_ff1 = nn.Linear(dimensions, dimensions * 4)
        self.w_ff2 = nn.Linear(dimensions * 4, dimensions)  

        self.norm1 = nn.LayerNorm(dimensions)
        self.norm2 = nn.LayerNorm(dimensions)

        self.fc = nn.Linear(dimensions, vocab_size)

    def forward(self, x):
        B , T = x.shape

        position = torch.arange(0, T, device=x.device).unsqueeze(0).expand(B, T)
        x = self.embedding(x) + self.pos_embedding(position)

        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)

        attention_score = torch.matmul( Q, K.transpose(-2, -1)) / (self.dimensions ** 0.5)

        mask = torch.tril(torch.ones(T, T)).to(attention_score.device)
        attention_score = attention_score.masked_fill(mask == 0, float('-inf'))

        attention_norm = torch.softmax(attention_score, dim=-1)
        out = torch.matmul(attention_norm, V)

        x = self.norm1(out + x)

        ff_hidden = self.w_ff1(x)
        ff_hidden = torch.relu(ff_hidden)
        ff_out = self.w_ff2(ff_hidden)

        x = self.norm2(ff_out + x)
        logits = self.fc(x)
        return logits




In [29]:
def get_batch(text, batch_size, seq_len):
    inputs = []
    targets = []

    for _ in range(batch_size):
        start_idx = np.random.randint(0, len(text) - seq_len - 1)
        chunk_x = text[start_idx : start_idx + seq_len]
        chunk_y = text[start_idx + 1 : start_idx + seq_len + 1]

        encoded_x = [char_to_idx[char] for char in chunk_x]
        encoded_y = [char_to_idx[char] for char in chunk_y]

        inputs.append(encoded_x)
        targets.append(encoded_y)
    return torch.tensor(inputs, dtype=torch.long), torch.tensor(targets, dtype=torch.long)

In [23]:
def encode(text):
    return [char_to_idx[char] for char in text if char in char_to_idx]
def decode(indices):
    return ''.join([idx_to_char[idx] for idx in indices])

In [35]:
def generate_text(model, prompt, max_length):
    model.eval()
    curr_seq = encode(prompt.lower())
    print("Prompt:", prompt)

    with torch.no_grad():
        for _ in range(max_length):
            context = curr_seq[-model.seq_len:]
            x = torch.tensor([context], dtype=torch.long)
            logits = model(x)
            last_char_logits = logits[0, -1, :]
            last_char_logits = last_char_logits / 0.8
            probs = torch.softmax(last_char_logits, dim=-1)
            next_char_idx = torch.multinomial(probs, num_samples=1).item()
            curr_seq.append(next_char_idx)
            print(idx_to_char[next_char_idx], end='', flush=True)
    print("\n")



In [37]:
with open('shakespeare.txt', 'r') as f:
    text = f.read().lower()

chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"Unique characters: {vocab_size, chars}")

char_to_idx = {ch: idx for idx, ch in enumerate(chars)}
idx_to_char = {idx: ch for idx, ch in enumerate(chars)}

embed_size = 128
seq_len = 32
batch_size = 64

model = Decoder(vocab_size=vocab_size, dimensions=embed_size, seq_len=seq_len)
print(model)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# Training --------------

for step in range(1000):
    X, Y = get_batch(text, batch_size, seq_len)
    logits = model(X)

    loss = criterion(logits.view(-1, vocab_size), Y.view(-1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(f"Step {step}, Loss: {loss.item():.4f}")
print("Training complete.")

print("--- Let's write a play! ---")
generate_text(model, prompt="romeo:\n", max_length=1000)


Unique characters: (58, ['\n', ' ', '!', '"', '&', "'", '(', ')', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '<', '>', '?', '[', ']', '_', '`', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '|', '}'])
Decoder(
  (embedding): Embedding(58, 128)
  (pos_embedding): Embedding(32, 128)
  (query): Linear(in_features=128, out_features=128, bias=True)
  (key): Linear(in_features=128, out_features=128, bias=True)
  (value): Linear(in_features=128, out_features=128, bias=True)
  (w_ff1): Linear(in_features=128, out_features=512, bias=True)
  (w_ff2): Linear(in_features=512, out_features=128, bias=True)
  (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (fc): Linear(in_features=128, out_features=58, bias=True)
)
Step 0, Loss: 4.2086
Step 100, Loss: 2.3067
Step 200, Loss: 2.1833
Step 300, Loss: 2.0062
St

In [5]:
!pip install transformers datasets


   ---------------------------------------- 0.0/526.6 kB ? eta -:--:--
   ------------------- -------------------- 262.1/526.6 kB ? eta -:--:--
   ---------------------------------------- 526.6/526.6 kB 1.9 MB/s  0:00:00
   ---------------------------------------- 0.0/27.5 MB ? eta -:--:--
   -- ------------------------------------- 1.8/27.5 MB 8.4 MB/s eta 0:00:04
   --- ------------------------------------ 2.1/27.5 MB 6.9 MB/s eta 0:00:04
   --- ------------------------------------ 2.1/27.5 MB 6.9 MB/s eta 0:00:04
   --- ------------------------------------ 2.1/27.5 MB 6.9 MB/s eta 0:00:04
   --- ------------------------------------ 2.4/27.5 MB 2.3 MB/s eta 0:00:11
   --- ------------------------------------ 2.4/27.5 MB 2.3 MB/s eta 0:00:11
   ---- ----------------------------------- 2.9/27.5 MB 2.0 MB/s eta 0:00:13
   ---- ----------------------------------- 2.9/27.5 MB 2.0 MB/s eta 0:00:13
   ---- ----------------------------------- 3.1/27.5 MB 1.7 MB/s eta 0:00:15
   ----- -------

In [ ]:
from transformers import AutoTokenizer

